<hr style="border:2px solid #0281c9"> </hr>

<img align="left" alt="ESO Logo" src="http://archive.eso.org/i/esologo.png">  

<div align="center">
  <h1 style="color: #0281c9; font-weight: bold;">ESO Science Archive</h1> 
  <h2 style="color: #0281c9; font-weight: bold;">Jupyter Notebooks</h2>
</div>

<hr style="border:2px solid #0281c9"> </hr>

# **Quick Start**

This notebook gives a compact first workflow for using `astroquery.eso` with the ESO Science Archive. It shows how to initialize the ESO interface, run a small positional search for raw and reduced observations, optionally download the matching products, and query a published catalogue table.

**Note:** Catalogue and TAP-backed archive access require an `astroquery.eso` version with the newer TAP support, such as `astroquery` 0.4.12 or later. If a query method used below is unavailable in your environment, update `astroquery` before continuing.

The later notebooks in this folder provide more detailed examples for authentication, raw observations, reduced products, TAP/ADQL queries, catalogue searches, and data downloads.

<hr style="border:2px solid #0281c9"> </hr>

# **Importing and basic usage of astroquery.eso**

We start by checking the installed `astroquery` version and creating an `Eso` instance. The `Eso` object stores the archive configuration used by the query and download helpers.

In [1]:
import astroquery # import astroquery
print(f"astroquery version: {astroquery.__version__}") # check the version of astroquery

astroquery version: 0.4.12.dev505+gf2a77a615.d20260427


In [2]:
from astroquery.eso import Eso # import the ESO module from astroquery

from astropy.coordinates import SkyCoord # sky coordinates and name resolution
import astropy.units as u # astronomical units
from pathlib import Path # filesystem paths

In [3]:
eso = Eso() # create an instance of the ESO class

By default, archive queries protect users from accidentally retrieving very large tables. 

For this quick start, we make the limit even smaller so that each query returns only a few representative rows.

In [4]:
eso.ROW_LIMIT = 5 # keep quick-start query results compact
print(f"Maximum number of records to return: {eso.ROW_LIMIT}")

Maximum number of records to return: 5


The observation examples below work with the public raw and reduced query helpers. The catalogue example at the end needs the newer catalogue interface, so we check for it before trying to use it.

In [5]:
has_catalogue_support = hasattr(eso, "query_catalog")

if not has_catalogue_support:
    print("Catalogue queries require an astroquery.eso version with query_catalog support, such as astroquery 0.4.12 or later.")

# **Search for observations**

Observation searches can target either raw archive metadata or reduced Phase 3 products. In both cases, a cone search uses right ascension, declination, and radius in degrees.

Here we resolve the star `HD 37903` and define a small 1 arcsec search radius. Name resolution uses online services, so the cell requires network access.

In [6]:
coords = SkyCoord.from_name("HD 37903") # resolve target name to coordinates
radius = 1 * u.arcsec # cone-search radius

ra = coords.ra.value # right ascension in degrees
dec = coords.dec.value # declination in degrees
cone_radius = radius.to(u.deg).value # radius in degrees

print(f"RA: {ra:.6f} deg")
print(f"Dec: {dec:.6f} deg")
print(f"Radius: {cone_radius:.8f} deg")

RA: 85.409955 deg
Dec: -2.259022 deg
Radius: 0.00027778 deg


## **Query raw observations**

Raw observations are the unprocessed observational data products acquired at the telescope and archived by ESO. The `query_main()` helper searches the general raw archive table; passing `"ESPRESSO"` restricts this example to [ESPRESSO](https://www.eso.org/sci/facilities/paranal/instruments/espresso.html) observations near the target position.

In [7]:
table_raw = eso.query_main(instruments="ESPRESSO", cone_ra=ra, cone_dec=dec, cone_radius=cone_radius)

table_raw

object,ra,dec,dp_id,date_obs,prog_id,access_estsize,access_url,datalink_url,dec_pnt,det_chip1id,det_chop_ncycles,det_dit,det_expid,det_ndit,dp_cat,dp_tech,dp_type,ecl_lat,ecl_lon,exp_start,exposure,filter_path,gal_lat,gal_lon,grat_path,gris_path,ins_mode,instrument,lambda_max,lambda_min,last_mod_date,mjd_obs,ob_id,ob_name,obs_mode,origfile,period,pi_coi,prog_title,prog_type,ra_pnt,release_date,s_region,slit_path,target,tel_airm_end,tel_airm_start,tel_alt,tel_ambi_fwhm_end,tel_ambi_fwhm_start,tel_ambi_pres_end,tel_ambi_pres_start,tel_ambi_rhum,tel_az,telescope,tpl_expno,tpl_id,tpl_name,tpl_nexp,tpl_seqno,tpl_start
,deg,deg,,,,kbyte,,,deg,,,s,,,,,,deg,deg,,s,,deg,deg,,,,,nm,nm,,d,,,,,,,,,deg,,,,,,,deg,arcsec,arcsec,hPa,hPa,%,deg,,,,,,,
object,float64,float64,object,object,object,int64,object,object,float64,object,int16,float32,int16,int16,object,object,object,float64,float64,object,float32,object,float64,float64,object,object,object,object,float64,float64,object,float64,int32,object,object,object,int16,object,object,int32,float64,object,object,object,object,float32,float32,float32,float32,float32,float32,float32,float32,float32,object,int32,object,object,int32,int32,object
HD 37903,85.41019194,-2.25911,ESPRE.2020-12-24T02:38:38.920,2020-12-24T02:38:38.920,106.20Y7.001,151806,https://dataportal.eso.org/dataPortal/file/ESPRE.2020-12-24T02:38:38.920,https://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ESPRE.2020-12-24T02:38:38.920,-2.25911,,--,--,22,1,SCIENCE,ECHELLE,"OBJECT,SKY",-25.617371,84.912548,2020-12-24T02:38:38.920Z,2900.0,FSND20,-16.537351,206.851485,,,SINGLEUHR,ESPRESSO,--,--,2020-12-24T03:35:57.180Z,59207.11017269,2866156,HD-37903,s,ESPRESSO_SINGLEUHR_OBS359_0001.fits,106,DE CIA/ FOX/ JENKINS/ LALLEMENT/ LEDOUX/ PETITJEAN/ SMOKER/ RAMBURUTH-HURT,LOCAL VARIATIONS OF METALLICITY IN THE NEUTRAL ISM: DISSECTING THE ORION REGION.,0,85.410192,2021-12-24T03:35:57Z,POSITION J2000 85.410192 -2.25911,,HD 37903,--,--,--,--,--,--,--,--,--,ESO-VLT-U2,1,ESPRESSO_singleUHR_obs,ESPRESSO_singleUHR_obs,1,2,2020-12-24T02:38:04


## **Query reduced observations**

Reduced observations are processed Phase 3 products contributed by ESO or the community. The `query_surveys()` helper searches these science products and returns identifiers that can be passed to `retrieve_data()`.

In [8]:
table_reduced = eso.query_surveys(surveys="ESPRESSO", cone_ra=ra, cone_dec=dec, cone_radius=cone_radius)

table_reduced

target_name,s_ra,s_dec,dp_id,proposal_id,abmaglim,access_estsize,access_format,access_url,bib_reference,calib_level,dataproduct_subtype,dataproduct_type,em_max,em_min,em_res_power,em_xel,facility_name,filter,gal_lat,gal_lon,instrument_name,is_solar,last_mod_date,multi_ob,n_obs,o_calib_status,o_ucd,obs_collection,obs_creator_did,obs_creator_name,obs_id,obs_publisher_did,obs_release_date,obs_title,obstech,p3orig,pol_states,pol_xel,preview_html,publication_date,release_description,s_fov,s_pixel_scale,s_region,s_resolution,s_xel1,s_xel2,snr,strehl,t_exptime,t_max,t_min,t_resolution,t_xel
,deg,deg,,,mag,kbyte,,,,,,,m,m,,,,,deg,deg,,,,,,,,,,,,,,,,,,,,,,deg,arcsec,,arcsec,,,,,s,d,d,s,
object,float64,float64,object,object,float64,int64,object,object,object,int32,object,object,float64,float64,float64,int64,object,object,float64,float64,object,int16,object,object,int32,object,object,object,object,object,object,object,object,object,object,object,object,int64,object,object,object,float64,float64,object,float64,int64,int64,float64,float64,float64,float64,float64,float64,int64
HD 37903,85.410192,-2.25911,ADP.2021-09-03T08:21:56.585,106.20Y7.001,--,55111,application/x-votable+xml;content=datalink,http://archive.eso.org/datalink/links?ID=ivo://eso.org/ID?ADP.2021-09-03T08:21:56.585,,2,flux-calibrated,spectrum,7.89997e-07,3.7719999999999996e-07,190000.0,443262,ESO-VLT-U2,,-16.537351,206.851485,ESPRESSO,0,2026-02-17T12:26:16.430Z,S,1,absolute,,ESPRESSO,ivo://eso.org/origfile?ES_SFLX_2866156_2020-12-24T02:38:38.920_UHR_1x1_U2.fits,"DE CIA, ANNALISA",2866156,ivo://eso.org/ID?ADP.2021-09-03T08:21:56.585,2021-12-24T03:35:57Z,HD37903_ES_SFLX_2866156_2020-12-24T02:38:38.920_UHR_1x1_U2,ECHELLE,IDP,,--,https://archive.eso.org/dataset/ADP.2021-09-03T08:21:56.585,2021-09-03T08:34:03Z,http://www.eso.org/rm/api/v1/public/releaseDescriptions/176,0.00013888,--,POSITION J2000 85.410192 -2.25911,--,--,--,292.9,--,2900.0,59207.1437375,59207.11017269,2899.999584,--


# **Download matching data products**

Both raw and reduced result tables include a `dp_id` column when downloadable products are available. The next cell keeps downloads disabled by default so that the quick start runs without writing archive products to disk. Set `download_data = True` to retrieve the matching products into the local `data/` directory.

In [9]:
download_data = False # set to True to download the matching products
download_dir = Path("data")

if download_data:
    download_dir.mkdir(exist_ok=True)

    raw_files = eso.retrieve_data(table_raw["dp_id"], destination=str(download_dir))
    reduced_files = eso.retrieve_data(table_reduced["dp_id"], destination=str(download_dir))

else:
    print("Downloads are disabled. Set download_data = True to retrieve files.")

Downloads are disabled. Set download_data = True to retrieve files.


# **Query a catalogue table**

Catalogue queries operate on published survey tables rather than observation products. A catalogue table contains column-based measurements, such as positions, magnitudes, classifications, or survey-specific quantities.

As a minimal example, we query a few rows from the KiDS DR4 catalogue. More detailed catalogue notebooks show how to inspect schemas, select columns, apply constraints, and perform positional searches.

In [10]:
table_cat = eso.query_catalog(catalog="KiDS_DR4_1_ugriZYJHKs_cat_fits")

table_cat

Level,ALPHA_J2000,A_IMAGE,A_WORLD,Agaper,Agaper_0p7,Agaper_1p0,BPZ_FILT,BPZ_FLAGFILT,BPZ_NONDETFILT,B_IMAGE,B_WORLD,BackGr,Bgaper,Bgaper_0p7,Bgaper_1p0,CHI_SQUARED_BPZ,CLASS_STAR,COLOUR_GAAP_H_Ks,COLOUR_GAAP_J_H,COLOUR_GAAP_Y_J,COLOUR_GAAP_Z_Y,COLOUR_GAAP_g_r,COLOUR_GAAP_i_Z,COLOUR_GAAP_r_i,COLOUR_GAAP_u_g,CXX_WORLD,CXY_WORLD,CYY_WORLD,DECJ2000,DELTA_J2000,ERRA_IMAGE,ERRA_WORLD,ERRB_IMAGE,ERRB_WORLD,ERRCXX_WORLD,ERRCXY_WORLD,ERRCYY_WORLD,ERRTHETA_J2000,ERRTHETA_WORLD,ERRX2_WORLD,ERRXY_WORLD,ERRY2_WORLD,EXTINCTION_H,EXTINCTION_J,EXTINCTION_Ks,EXTINCTION_Y,EXTINCTION_Z,EXTINCTION_g,EXTINCTION_i,EXTINCTION_r,EXTINCTION_u,FIELD_POS,FLAG_GAAP_0p7_H,FLAG_GAAP_0p7_J,FLAG_GAAP_0p7_Ks,FLAG_GAAP_0p7_Y,FLAG_GAAP_0p7_Z,FLAG_GAAP_0p7_g,FLAG_GAAP_0p7_i,FLAG_GAAP_0p7_r,FLAG_GAAP_0p7_u,FLAG_GAAP_1p0_H,FLAG_GAAP_1p0_J,FLAG_GAAP_1p0_Ks,FLAG_GAAP_1p0_Y,FLAG_GAAP_1p0_Z,FLAG_GAAP_1p0_g,FLAG_GAAP_1p0_i,FLAG_GAAP_1p0_r,FLAG_GAAP_1p0_u,FLAG_GAAP_H,FLAG_GAAP_J,FLAG_GAAP_Ks,FLAG_GAAP_Y,FLAG_GAAP_Z,FLAG_GAAP_g,FLAG_GAAP_i,FLAG_GAAP_r,FLAG_GAAP_u,FLUXERR_APER_10,FLUXERR_APER_100,FLUXERR_APER_14,FLUXERR_APER_20,FLUXERR_APER_30,FLUXERR_APER_4,FLUXERR_APER_40,FLUXERR_APER_6,FLUXERR_APER_60,FLUXERR_APER_8,FLUXERR_AUTO,FLUXERR_GAAP_0p7_H,FLUXERR_GAAP_0p7_J,FLUXERR_GAAP_0p7_Ks,FLUXERR_GAAP_0p7_Y,FLUXERR_GAAP_0p7_Z,FLUXERR_GAAP_0p7_g,FLUXERR_GAAP_0p7_i,FLUXERR_GAAP_0p7_r,FLUXERR_GAAP_0p7_u,FLUXERR_GAAP_1p0_H,FLUXERR_GAAP_1p0_J,FLUXERR_GAAP_1p0_Ks,FLUXERR_GAAP_1p0_Y,FLUXERR_GAAP_1p0_Z,FLUXERR_GAAP_1p0_g,FLUXERR_GAAP_1p0_i,FLUXERR_GAAP_1p0_r,FLUXERR_GAAP_1p0_u,FLUXERR_GAAP_H,FLUXERR_GAAP_J,FLUXERR_GAAP_Ks,FLUXERR_GAAP_Y,FLUXERR_GAAP_Z,FLUXERR_GAAP_g,FLUXERR_GAAP_i,FLUXERR_GAAP_r,FLUXERR_GAAP_u,FLUXERR_ISO,FLUXERR_ISOCOR,FLUX_APER_10,FLUX_APER_100,FLUX_APER_14,FLUX_APER_20,FLUX_APER_30,FLUX_APER_4,FLUX_APER_40,FLUX_APER_6,FLUX_APER_60,FLUX_APER_8,FLUX_AUTO,FLUX_GAAP_0p7_H,FLUX_GAAP_0p7_J,FLUX_GAAP_0p7_Ks,FLUX_GAAP_0p7_Y,FLUX_GAAP_0p7_Z,FLUX_GAAP_0p7_g,FLUX_GAAP_0p7_i,FLUX_GAAP_0p7_r,FLUX_GAAP_0p7_u,FLUX_GAAP_1p0_H,FLUX_GAAP_1p0_J,FLUX_GAAP_1p0_Ks,FLUX_GAAP_1p0_Y,FLUX_GAAP_1p0_Z,FLUX_GAAP_1p0_g,FLUX_GAAP_1p0_i,FLUX_GAAP_1p0_r,FLUX_GAAP_1p0_u,FLUX_GAAP_H,FLUX_GAAP_J,FLUX_GAAP_Ks,FLUX_GAAP_Y,FLUX_GAAP_Z,FLUX_GAAP_g,FLUX_GAAP_i,FLUX_GAAP_r,FLUX_GAAP_u,FLUX_ISO,FLUX_ISOCOR,FLUX_RADIUS,FWHM_IMAGE,FWHM_WORLD,Flag,HTM,ID,IMAFLAGS_ISO,ISO0,ISO1,ISO2,ISO3,ISO4,ISO5,ISO6,ISO7,ISOAREA_WORLD,KIDS_TILE,KRON_RADIUS,MAGERR_APER_10,MAGERR_APER_100,MAGERR_APER_14,MAGERR_APER_20,MAGERR_APER_4,MAGERR_APER_40,MAGERR_APER_6,MAGERR_APER_60,MAGERR_APER_8,MAGERR_AUTO,MAGERR_GAAP_0p7_H,MAGERR_GAAP_0p7_J,MAGERR_GAAP_0p7_Ks,MAGERR_GAAP_0p7_Y,MAGERR_GAAP_0p7_Z,MAGERR_GAAP_0p7_g,MAGERR_GAAP_0p7_i,MAGERR_GAAP_0p7_r,MAGERR_GAAP_0p7_u,MAGERR_GAAP_1p0_H,MAGERR_GAAP_1p0_J,MAGERR_GAAP_1p0_Ks,MAGERR_GAAP_1p0_Y,MAGERR_GAAP_1p0_Z,MAGERR_GAAP_1p0_g,MAGERR_GAAP_1p0_i,MAGERR_GAAP_1p0_r,MAGERR_GAAP_1p0_u,MAGERR_GAAP_H,MAGERR_GAAP_J,MAGERR_GAAP_Ks,MAGERR_GAAP_Y,MAGERR_GAAP_Z,MAGERR_GAAP_g,MAGERR_GAAP_i,MAGERR_GAAP_r,MAGERR_GAAP_u,MAGERR_ISO,MAGERR_ISOCOR,MAGGERR_APER_30,MAG_APER_10,MAG_APER_100,MAG_APER_14,MAG_APER_20,MAG_APER_30,MAG_APER_4,MAG_APER_40,MAG_APER_6,MAG_APER_60,MAG_APER_8,MAG_AUTO,MAG_GAAP_0p7_H,MAG_GAAP_0p7_J,MAG_GAAP_0p7_Ks,MAG_GAAP_0p7_Y,MAG_GAAP_0p7_Z,MAG_GAAP_0p7_g,MAG_GAAP_0p7_i,MAG_GAAP_0p7_r,MAG_GAAP_0p7_u,MAG_GAAP_1p0_H,MAG_GAAP_1p0_J,MAG_GAAP_1p0_Ks,MAG_GAAP_1p0_Y,MAG_GAAP_1p0_Z,MAG_GAAP_1p0_g,MAG_GAAP_1p0_i,MAG_GAAP_1p0_r,MAG_GAAP_1p0_u,MAG_GAAP_H,MAG_GAAP_J,MAG_GAAP_Ks,MAG_GAAP_Y,MAG_GAAP_Z,MAG_GAAP_g,MAG_GAAP_i,MAG_GAAP_r,MAG_GAAP_u,MAG_ISO,MAG_ISOCOR,MAG_LIM_H,MAG_LIM_J,MAG_LIM_Ks,MAG_LIM_Y,MAG_LIM_Z,MAG_LIM_g,MAG_LIM_i,MAG_LIM_r,MAG_LIM_u,MASK,MU_MAX,MU_THRESHOLD,M_0,MaxVal,NBPZ_FILT,NBPZ_FLAGFILT,NBPZ_NONDETFILT,NIMAFLAGS_ISO,ODDS,PAgaap,RAJ2000,SG2DPHOT,SG_FLAG,SID,SLID,S_ELLIPTICITY,S_ELONGATION,SeqNr,THELI_NAME,THETA_J2000,THETA_WORLD,T_B,T_ML,X2_WORLD,XMAX_IMAGE,XMIN_IMAGE,XY_WORLD,X_WORLD,Xpos,Y2_WORLD,YMAX_IMAGE,YMIN_IMAGE,Y_WORLD,Ypos,Z_B,Z_B_MAX,Z_B_MIN,Z_ML
count,deg,p

The `ROW_LIMIT` attribute controls how many matching catalogue rows are returned. Use a small value while exploring a table. To disable truncation, set `eso.ROW_LIMIT = None`, `0`, or `-1`, but only for queries that are constrained enough to return a safe amount of data.

<hr style="border:2px solid #0281c9"> </hr>